In [ ]:
import os
import re
import pandas as pd
import glob

def count_line_types(file_path):
    code_lines = 0
    comment_lines = 0
    empty_lines = 0
    in_block_comment = False

    with open(file_path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            stripped = line.strip()
            if not stripped:
                empty_lines += 1
                continue

            if in_block_comment:
                comment_lines += 1
                if '*/' in stripped:
                    in_block_comment = False
                continue

            if stripped.startswith('/*'):
                comment_lines += 1
                if not '*/' in stripped:
                    in_block_comment = True
                continue

            if stripped.startswith('//'):
                comment_lines += 1
                continue

            if '/*' in stripped:
                before_comment = stripped.split('/*', 1)[0].strip()
                if before_comment:
                    code_lines += 1
                else:
                    comment_lines += 1
                if not '*/' in stripped:
                    in_block_comment = True
                continue

            if '//' in stripped:
                before_comment = stripped.split('//', 1)[0].strip()
                if before_comment:
                    code_lines += 1
                else:
                    comment_lines += 1
                continue

            code_lines += 1

    return code_lines, comment_lines, empty_lines

def count_test_methods(content):
    test_annotation_pattern = re.compile(
        r'@(?:org\.junit(?:\.jupiter\.api)?\.)?(Test|RepeatedTest|ParameterizedTest)\b'
    )
    method_pattern = re.compile(
        r'\b\w[\w<>\[\],\s]*\s+\w+\s*\('
    )

    lines = content.splitlines()
    num_test_methods = 0
    pending_test_method = False
    method_decl = ''
    for line in lines:
        stripped = line.strip()
        # Ignore single-line comments and JavaDoc lines
        if stripped.startswith('//') or stripped.startswith('*') or stripped.startswith('/*') or stripped.startswith('*/'):
            continue

        # If annotation and method on the same line
        if test_annotation_pattern.search(stripped) and method_pattern.search(stripped):
            num_test_methods += 1
            pending_test_method = False
            method_decl = ''
            continue

        # If we see a test annotation, set the flag
        if test_annotation_pattern.search(stripped):
            pending_test_method = True
            method_decl = ''
            continue

        # If we're waiting for a method after a test annotation
        if pending_test_method:
            # Skip blank lines and annotation lines
            if not stripped or stripped.startswith('@') or stripped.startswith('//') or stripped.startswith('*') or stripped.startswith('/*') or stripped.startswith('*/'):
                continue
            # Accumulate lines for multi-line method declarations
            method_decl += ' ' + stripped
            if '(' in method_decl:
                if method_pattern.search(method_decl):
                    num_test_methods += 1
                    pending_test_method = False
                    method_decl = ''
            continue

        # Reset method_decl if not in pending state
        method_decl = ''
    return num_test_methods

def collect_stats(directory):
    num_files = 0
    num_classes = 0
    num_code_lines = 0
    num_comment_lines = 0
    num_empty_lines = 0
    num_test_methods = 0

    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".java"):
                num_files += 1
                file_path = os.path.join(root, file)
                code, comment, empty = count_line_types(file_path)
                num_code_lines += code
                num_comment_lines += comment
                num_empty_lines += empty

                with open(file_path, encoding="utf-8", errors="ignore") as f:
                    content = f.read()
                    num_classes += len(re.findall(r'\b(class|enum)\s+\w+', content))
                    num_test_methods += count_test_methods(content)

    return {
        "directory": directory,
        "num_files": num_files,
        "num_classes": num_classes,
        "num_code_lines": num_code_lines,
        "num_comment_lines": num_comment_lines,
        "num_empty_lines": num_empty_lines,
        "num_test_methods": num_test_methods,
    }

# List of directories to analyze
directories = [
    "../projects/commons-utils/src/main",
    "../projects/commons-utils/src/test",
    "../projects/commons-utils-es-default-1s/src/main",
    "../projects/commons-utils-es-default-1s/src/test",
    "../projects/commons-utils-es-default-10s/src/main",
    "../projects/commons-utils-es-default-10s/src/test",
    "../projects/commons-utils-es-default-60s/src/main",
    "../projects/commons-utils-es-default-60s/src/test",
    "../projects/eqbench-es-default-1s/src/main",
    "../projects/eqbench-es-default-1s/src/test",
    "../projects/eqbench-es-default-10s/src/main",
    "../projects/eqbench-es-default-10s/src/test",
    "../projects/eqbench-es-default-60s/src/main",
    "../projects/eqbench-es-default-60s/src/test",
]

# Helper to extract project name from directory path
def get_project_name(directory):
    # Assumes project is the last segment before 'src'
    parts = directory.split(os.sep)
    for i, part in enumerate(parts):
        if part == "src" and i > 0:
            return parts[i-1]
    return directory

# Discover github_com_* projects and add their src/main and src/test ---
github_projects_root = "../projects"
github_project_dirs = glob.glob(os.path.join(github_projects_root, "github_com_*"))

for project_dir in github_project_dirs:
    main_dir = os.path.join(project_dir, "src", "main")
    test_dir = os.path.join(project_dir, "src", "test")
    if os.path.isdir(main_dir):
        directories.append(main_dir)
    if os.path.isdir(test_dir):
        directories.append(test_dir)

# Collect statistics for all directories
stats = []
for d in directories:
    stat = collect_stats(d)
    stat["project"] = get_project_name(d)
    stat["type"] = "main" if d.endswith("/main") else "test"
    stats.append(stat)

df = pd.DataFrame(stats)
display(df)

# Aggregate by project
summary = []

# --- AGGREGATE: All non-github projects individually ---
non_github_df = df[~df["project"].str.startswith("github_com_")]
for project in non_github_df["project"].unique():
    main = non_github_df[(non_github_df["project"] == project) & (non_github_df["type"] == "main")]
    test = non_github_df[(non_github_df["project"] == project) & (non_github_df["type"] == "test")]
    summary.append({
        "project": project,
        "main_files": int(main["num_files"].sum()),
        "main_classes": int(main["num_classes"].sum()),
        "main_sloc": int(main["num_code_lines"].sum()),
        "test_files": int(test["num_files"].sum()),
        "test_classes": int(test["num_classes"].sum()),
        "test_sloc": int(test["num_code_lines"].sum()),
        "test_methods": int(test["num_test_methods"].sum()),
        "is_github": False,
    })

# --- AGGREGATE: github_com_* projects (total, mean, median) ---
github_df = df[df["project"].str.startswith("github_com_")]

if not github_df.empty:
    # Group by project and type
    github_projects = github_df["project"].unique()
    github_rows = []
    for project in github_projects:
        main = github_df[(github_df["project"] == project) & (github_df["type"] == "main")]
        test = github_df[(github_df["project"] == project) & (github_df["type"] == "test")]
        github_rows.append({
            "main_files": int(main["num_files"].sum()),
            "main_classes": int(main["num_classes"].sum()),
            "main_sloc": int(main["num_code_lines"].sum()),
            "test_files": int(test["num_files"].sum()),
            "test_classes": int(test["num_classes"].sum()),
            "test_sloc": int(test["num_code_lines"].sum()),
            "test_methods": int(test["num_test_methods"].sum()),
        })
    # Convert to DataFrame for easy stats
    github_stats = pd.DataFrame(github_rows)

    # Total
    summary.append({
        "project": "additional-rq4-oss (total)",
        "main_files": int(github_stats["main_files"].sum()),
        "main_classes": int(github_stats["main_classes"].sum()),
        "main_sloc": int(github_stats["main_sloc"].sum()),
        "test_files": int(github_stats["test_files"].sum()),
        "test_classes": int(github_stats["test_classes"].sum()),
        "test_sloc": int(github_stats["test_sloc"].sum()),
        "test_methods": int(github_stats["test_methods"].sum()),
        "is_github": True,
    })
    # Mean
    summary.append({
        "project": "additional-rq4-oss (mean)",
        "main_files": int(github_stats["main_files"].mean()),
        "main_classes": int(github_stats["main_classes"].mean()),
        "main_sloc": int(github_stats["main_sloc"].mean()),
        "test_files": int(github_stats["test_files"].mean()),
        "test_classes": int(github_stats["test_classes"].mean()),
        "test_sloc": int(github_stats["test_sloc"].mean()),
        "test_methods": int(github_stats["test_methods"].mean()),
        "is_github": True,
    })
    # Median
    summary.append({
        "project": "additional-rq4-oss (median)",
        "main_files": int(github_stats["main_files"].median()),
        "main_classes": int(github_stats["main_classes"].median()),
        "main_sloc": int(github_stats["main_sloc"].median()),
        "test_files": int(github_stats["test_files"].median()),
        "test_classes": int(github_stats["test_classes"].median()),
        "test_sloc": int(github_stats["test_sloc"].median()),
        "test_methods": int(github_stats["test_methods"].median()),
        "is_github": True,
    })

df_summary = pd.DataFrame(summary)

# --- Build LaTeX table ---
def get_base_project(name):
    if name.startswith("additional-rq4-oss"):
        return "additional-rq4-oss"
    return name.split('-es-')[0]

def fmt(x):
    return f"{x:,}"

latex_table = r"""\begin{table}[H]
  \caption{Number of files, classes, source lines of code (SLOC), and test methods per project.}
  \label{tab:dataset-statistics}
  \begin{tabular}{lrrrrrrr}
    \toprule
    & \multicolumn{3}{r}{Implementation} & \multicolumn{4}{r}{Test} \\
    \cmidrule(lr){2-4} \cmidrule(lr){5-8}
    Project & Files & Classes & SLOC & Files & Classes & SLOC & Methods \\
    \midrule
"""

# First, non-github projects (sorted/grouped as before)
non_github = df_summary[~df_summary["is_github"]]
prev_base = None
for _, row in non_github.iterrows():
    base = get_base_project(row["project"])
    if prev_base is not None and base != prev_base:
        latex_table += "    \\midrule\n"
    prev_base = base
    latex_table += (
        f"    {row['project']} & "
        f"{fmt(row['main_files'])} & {fmt(row['main_classes'])} & {fmt(row['main_sloc'])} & "
        f"{fmt(row['test_files'])} & {fmt(row['test_classes'])} & {fmt(row['test_sloc'])} & {fmt(row['test_methods'])} \\\\\n"
    )

# Then, github summary rows (always at the bottom, no midrule)
github = df_summary[df_summary["is_github"]]
if not github.empty:
    latex_table += "    \\midrule\n"
    for _, row in github.iterrows():
        latex_table += (
            f"    {row['project']} & "
            f"{fmt(row['main_files'])} & {fmt(row['main_classes'])} & {fmt(row['main_sloc'])} & "
            f"{fmt(row['test_files'])} & {fmt(row['test_classes'])} & {fmt(row['test_sloc'])} & {fmt(row['test_methods'])} \\\\\n"
        )

latex_table += r"""    \bottomrule
  \end{tabular}
\end{table}
"""

print(latex_table)
